# ToT 单事件 TXT → 图片（论文用）

本 Notebook 用于：将单事件 ToT 像素矩阵 txt（例如 100×100，空格分隔，0.000000 为未激活）批量转换为单张图片文件。

**满足你的约束**
- 不做轨迹提取：每个 txt 视为一个单事件。
- `box_size` 可调（正方形），事件自动居中。
- 若事件外接框尺寸大于 `box_size`：接受裁切，并在控制台提示。
- ToT→颜色：线性映射，**全局固定** `vmin/vmax`；0 值 mask 成白色/背景。
- 每个事件输出**单独** PNG；每张图都有 colorbar；不合并子图。

In [10]:
from __future__ import annotations

from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

# ======================
# 你需要改的参数区
# ======================
# 输入：单事件 txt 所在目录（可递归）
INPUT_DIR = Path(r"E:\C1Analysis\Special")
RECURSIVE = True
PATTERN = "*.txt"

# 输出：图片保存目录（作为输出根目录）
OUTPUT_DIR = Path(r"E:\C1Analysis\figure")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# 是否保持与原数据目录一致的层级结构：
# 例如 INPUT_DIR/a/b/evt.txt  ->  OUTPUT_DIR/a/b/evt_boxXX.png
MIRROR_INPUT_TREE = True

# 事件居中后的 box（正方形）
BOX_SIZE = 32  # 例如 32/64/100

# 激活阈值：>THRESHOLD 认为是激活像素
THRESHOLD = 0.0

# 颜色映射（与你学长一致用 turbo）
CMAP = "turbo"

# ToT → 颜色的全局固定范围（你给的激活像素约 500–1000）
# 建议：vmin 取激活像素下界，vmax 取激活像素上界，这样对比度最好。
VMIN = 0
VMAX = 500

# 导出图片参数
DPI = 600
COLORBAR_ORIENTATION = "horizontal"  # 你的要求：每张图都要 colorbar

# 是否在图上显示文件名（论文最终版通常设 False）
SHOW_TITLE = False

print("Config loaded.")

Config loaded.


In [11]:
def load_tot_txt(file_path: Path) -> np.ndarray:
    data = np.loadtxt(file_path)
    if data.ndim != 2:
        raise ValueError(f"Not a 2D matrix: {file_path} (ndim={data.ndim})")
    if data.shape[0] != data.shape[1]:
        raise ValueError(f"Not a square matrix: {file_path} (shape={data.shape})")
    return data


def bbox_from_mask(mask: np.ndarray) -> tuple[int, int, int, int]:
    """Return (ymin, ymax, xmin, xmax) inclusive."""
    rows = np.any(mask, axis=1)
    cols = np.any(mask, axis=0)
    y_indices = np.where(rows)[0]
    x_indices = np.where(cols)[0]
    if y_indices.size == 0 or x_indices.size == 0:
        raise ValueError("Empty mask: no active pixels.")
    ymin, ymax = int(y_indices[0]), int(y_indices[-1])
    xmin, xmax = int(x_indices[0]), int(x_indices[-1])
    return ymin, ymax, xmin, xmax


def center_event_in_box(
    data: np.ndarray,
    mask: np.ndarray,
    box_size: int,
) -> tuple[np.ndarray, bool, tuple[int, int, int, int]]:
    """
    将事件（mask 的外接框区域）裁剪出来并放入 box_size×box_size 中心。

    若外接框尺寸大于 box_size，会以外接框中心为基准裁切到 box_size，并返回 cropped=True。
    """
    ymin, ymax, xmin, xmax = bbox_from_mask(mask)
    h = ymax - ymin + 1
    w = xmax - xmin + 1

    cropped_flag = False

    # 目标区域（从原图上截取）
    if h > box_size or w > box_size:
        cropped_flag = True
        cy = (ymin + ymax) // 2
        cx = (xmin + xmax) // 2
        half = box_size // 2
        # 取 box_size 的窗口；注意 box_size 奇偶时的对齐
        y0 = cy - half
        x0 = cx - half
        y1 = y0 + box_size
        x1 = x0 + box_size

        # 边界裁剪到原图范围
        y0_clamped = max(0, y0)
        x0_clamped = max(0, x0)
        y1_clamped = min(data.shape[0], y1)
        x1_clamped = min(data.shape[1], x1)

        window = data[y0_clamped:y1_clamped, x0_clamped:x1_clamped]

        out = np.zeros((box_size, box_size), dtype=data.dtype)
        # 将 window 放到 out 的对应位置（如果窗口被 clamp，会有偏移）
        oy0 = y0_clamped - y0
        ox0 = x0_clamped - x0
        out[oy0:oy0 + window.shape[0], ox0:ox0 + window.shape[1]] = window
        return out, cropped_flag, (ymin, ymax, xmin, xmax)

    # 不需要裁切：把外接框内容居中放入 box
    patch = data[ymin:ymax + 1, xmin:xmax + 1]
    out = np.zeros((box_size, box_size), dtype=data.dtype)
    y_offset = (box_size - h) // 2
    x_offset = (box_size - w) // 2
    out[y_offset:y_offset + h, x_offset:x_offset + w] = patch
    return out, cropped_flag, (ymin, ymax, xmin, xmax)


def mask_zeros_to_nan(arr: np.ndarray) -> np.ndarray:
    # 注意：NaN 需要 float 类型
    arr_f = arr.astype(float, copy=False)
    return np.where(arr_f == 0.0, np.nan, arr_f)

In [12]:
def iter_input_files(input_dir: Path, pattern: str, recursive: bool) -> list[Path]:
    if recursive:
        files = sorted(input_dir.rglob(pattern))
    else:
        files = sorted(input_dir.glob(pattern))
    return [p for p in files if p.is_file()]


def render_and_save_image(
    centered: np.ndarray,
    out_path: Path,
    *,
    cmap: str,
    vmin: float,
    vmax: float,
    dpi: int,
    colorbar_orientation: str = "horizontal",
    show_title: bool = False,
    title: str = "",
) -> None:
    data_for_plot = mask_zeros_to_nan(centered)

    fig, ax = plt.subplots(figsize=(3.2, 3.2))
    im = ax.imshow(
        data_for_plot,
        cmap=cmap,
        interpolation="nearest",
        vmin=vmin,
        vmax=vmax,
        extent=(0.0, float(centered.shape[1]), 0.0, float(centered.shape[0])),
    )

    ax.set_xlim(0, centered.shape[1])
    ax.set_ylim(0, centered.shape[0])
    ax.set_aspect("equal")
    ax.set_xticks([])
    ax.set_yticks([])

    for spine in ax.spines.values():
        spine.set_visible(True)
        spine.set_linewidth(1.5)

    if show_title and title:
        ax.set_title(title, fontsize=8)

    cbar = fig.colorbar(
        im,
        ax=ax,
        orientation=colorbar_orientation,
        pad=0.10 if colorbar_orientation == "horizontal" else 0.04,
        fraction=0.05,
    )
    cbar.set_label("ToT (a.u.)", fontsize=9)
    cbar.ax.tick_params(labelsize=8)

    fig.savefig(out_path, dpi=dpi, bbox_inches="tight", pad_inches=0.02)
    plt.close(fig)

In [13]:
files = iter_input_files(INPUT_DIR, PATTERN, RECURSIVE)
print(f"Found {len(files)} files under: {INPUT_DIR}")

if len(files) == 0:
    raise SystemExit("No input files found. Please check INPUT_DIR/PATTERN/RECURSIVE.")

n_skipped_empty = 0
n_cropped = 0

for idx, file_path in enumerate(files, start=1):
    data = load_tot_txt(file_path)
    mask = data > THRESHOLD

    if not np.any(mask):
        n_skipped_empty += 1
        print(f"[SKIP empty] {file_path}")
        continue

    centered, cropped_flag, bbox = center_event_in_box(data, mask, BOX_SIZE)
    if cropped_flag:
        n_cropped += 1
        ymin, ymax, xmin, xmax = bbox
        h = ymax - ymin + 1
        w = xmax - xmin + 1
        print(f"[CROP] {file_path.name}: bbox={h}x{w} > box={BOX_SIZE} (cropped to {BOX_SIZE}x{BOX_SIZE})")

    # 输出路径：可选择镜像输入目录结构
    try:
        rel = file_path.relative_to(INPUT_DIR)
    except ValueError:
        # 极少数情况下（例如传入了不同盘符/不在 INPUT_DIR 下）无法 relative_to
        rel = Path(file_path.name)

    out_dir = (OUTPUT_DIR / rel.parent) if MIRROR_INPUT_TREE else OUTPUT_DIR
    out_dir.mkdir(parents=True, exist_ok=True)

    out_name = f"{rel.stem}_box{BOX_SIZE}.png"
    out_path = out_dir / out_name

    render_and_save_image(
        centered,
        out_path,
        cmap=CMAP,
        vmin=VMIN,
        vmax=VMAX,
        dpi=DPI,
        colorbar_orientation=COLORBAR_ORIENTATION,
        show_title=SHOW_TITLE,
        title=file_path.stem,
    )

    if idx % 50 == 0:
        print(f"Processed {idx}/{len(files)}...")

print("Done.")
print(f"- saved images: {len(files) - n_skipped_empty}")
print(f"- skipped empty events: {n_skipped_empty}")
print(f"- cropped events: {n_cropped}")

Found 160 files under: E:\C1Analysis\Special
[CROP] C_r00000_0090_074.txt: bbox=8x52 > box=32 (cropped to 32x32)
Processed 50/160...
Processed 100/160...
Processed 150/160...
Done.
- saved images: 160
- skipped empty events: 0
- cropped events: 1


## ToT→颜色映射说明（论文写法可用）

- 本脚本采用线性伪彩色映射：将 ToT 数值按固定范围 $[\mathrm{VMIN},\ \mathrm{VMAX}]$ 线性归一化后映射到 colormap（默认 `turbo`）。
- 未激活像素（ToT=0）被 mask 为 NaN，以白色背景显示，不参与 colorbar 标定。
- 若存在 ToT 超出范围：>VMAX 的像素会颜色饱和；<VMIN 的像素会接近最低端颜色。你可以通过调整 `VMIN/VMAX` 改善对比度与可比性。